# 05 - Silver: `ecommerce_pedidos`
Lê da Bronze só o que ainda não foi processado, converte as datas, aplica as regras da planilha e separa o resultado em **Silver** (válidos) e **quarentena** (reprovados, com o motivo).
Cada regra vira uma coluna `flag_*`, para a quarentena explicar por que a linha foi barrada.

| Regra | Coluna | Critério |
|---|---|---|
| T1 | `flag_schema` | coluna obrigatória ausente no arquivo (100% nula no lote de origem) ou nula na linha |
| T2 | — | watermark por `bronze_ingested_at` + upsert por `id_pedido` |
| T3 | `flag_valor_total` | `valor_total` nulo ou ≤ 0 |
| T4 | `flag_dt_pedido` | `dt_pedido` não converte para data |
| T5 | `flag_status` | `status_pedido` fora da lista permitida |
| N10 | `flag_status_antes_pedido` | `dt_ultima_atualizacao_status` < `dt_pedido` |
| extra | `flag_frete` | `valor_frete` < 0 |

In [0]:
%run ./00_setup_config

## Parâmetros

In [0]:
from pyspark.sql import functions as F, Window

CATALOGO, GRUPO, FUSO_NEGOCIO = "workspace", "grupo2", "America/Sao_Paulo"
TABELA_BRONZE = f"{CATALOGO}.{SQUAD}.bronze_{TABELA}"
BASE = f"abfss://{SQUAD}@{_sufixo}/{GRUPO}".rstrip("/")
CAMINHO_SILVER, CAMINHO_QUARENTENA = f"{BASE}/silver/{TABELA}", f"{BASE}/quarantine/{TABELA}"

OBRIGATORIAS = ["id_pedido", "id_cliente", "dt_pedido", "status_pedido", "valor_total", "metodo_pagamento"]
# Lista provisória: status vistos no batch + "Aguardando" (só no tempo real). Pendente de confirmação do negócio.
STATUS_VALIDOS = ["Aguardando", "Processando", "Pagamento Aprovado", "Em Separação", "Enviado", "Entregue", "Cancelado"]
COLUNAS_DATA = ["dt_pedido", "dt_previsao_entrega", "dt_ultima_atualizacao_status"]

print(f"Bronze: {TABELA_BRONZE}\nSilver: {CAMINHO_SILVER}\nQuarentena: {CAMINHO_QUARENTENA}")

## Leitura incremental (watermark)
O watermark é o maior `bronze_ingested_at` já visto na Silver **ou** na quarentena; sem olhar a quarentena, um lote 100% reprovado seria reprocessado a cada execução.
As camadas ficam em pasta Delta no container da squad, por isso toda leitura e escrita leva `adls_options`.

In [0]:
def ler_delta(caminho):
    """Devolve o DataFrame Delta ou None se o caminho/tabela ainda não existe."""
    try:
        df = spark.read.format("delta").options(**adls_options).load(caminho)
        df.schema  # força a análise AGORA, dentro do try (Spark Connect é lazy)
        return df
    except Exception as e:
        msg = str(e)
        if any(t in msg for t in (
            "PATH_NOT_FOUND",
            "DELTA_PATH_DOES_NOT_EXIST",
            "DELTA_MISSING_DELTA_TABLE",
            "is not a Delta table",
            "does not exist",
        )):
            return None
        raise  # 401/403 e outros erros reais continuam estourando


silver_df, quarentena_df = ler_delta(CAMINHO_SILVER), ler_delta(CAMINHO_QUARENTENA)

marcas = [
    df.agg(F.max("bronze_ingested_at")).first()[0]
    for df in (silver_df, quarentena_df) if df is not None
]
watermark = max([m for m in marcas if m is not None], default=None)

bronze = spark.table(TABELA_BRONZE).drop("ano", "mes", "dia", "hora")
lote = bronze if watermark is None else bronze.filter(F.col("bronze_ingested_at") > watermark)
total_lote = lote.count()
print(f"Watermark: {watermark}  |  linhas novas na Bronze: {total_lote}")

## Conversão de datas
A Bronze guarda `dt_*` como inteiro em **nanossegundos**, exatamente como veio do gerador, e o valor representa o horário de Brasília (sem fuso).
`make_timestamp(..., FUSO_NEGOCIO)` monta o instante correto independente do fuso da sessão Spark, para a Gold poder comparar com `current_timestamp()`.

In [0]:
def nanos_para_timestamp(c):
    local = F.expr(f"TIMESTAMP_NTZ'1970-01-01 00:00:00' + make_dt_interval(0, 0, 0, CAST({c} / 1000000000 AS DECIMAL(18,6)))")
    seg = (F.col(c) % 60_000_000_000 / 1e9).cast("decimal(8,6)")
    return F.make_timestamp(F.year(local), F.month(local), F.dayofmonth(local), F.hour(local), F.minute(local), seg, F.lit(FUSO_NEGOCIO))

tratado = lote.select(*[nanos_para_timestamp(c).alias(c) if c in COLUNAS_DATA else F.col(c) for c in lote.columns])
tratado = (tratado.withColumn("status_pedido", F.trim("status_pedido"))
                  .withColumn("metodo_pagamento", F.trim("metodo_pagamento"))
                  .withColumn("silver_processed_at", F.current_timestamp()))

## Regras e flags
T1: a Bronze lê com schema fixo, então uma coluna ausente no arquivo chega como nula em todas as linhas daquele arquivo. Por isso a checagem é "coluna 100% nula no `bronze_source_file`", além de nula na linha.

In [0]:
por_arquivo = Window.partitionBy("bronze_source_file")
coluna_ausente = F.greatest(*[F.max(F.col(c).isNotNull().cast("int")).over(por_arquivo) == 0 for c in OBRIGATORIAS])
linha_nula = F.greatest(*[F.col(c).isNull() for c in OBRIGATORIAS])

regras = {
    "flag_schema":              coluna_ausente | linha_nula,
    "flag_valor_total":         F.col("valor_total").isNull() | (F.col("valor_total") <= 0),
    "flag_dt_pedido":           F.col("dt_pedido").isNull(),
    "flag_status":              ~F.coalesce(F.col("status_pedido").isin(STATUS_VALIDOS), F.lit(False)),
    "flag_status_antes_pedido": F.coalesce(F.col("dt_ultima_atualizacao_status") < F.col("dt_pedido"), F.lit(False)),
    "flag_frete":               F.coalesce(F.col("valor_frete") < 0, F.lit(False)),
}
validado = tratado.select("*", *[F.coalesce(cond, F.lit(True)).alias(nome) for nome, cond in regras.items()])
validado = validado.withColumn("quarantine_reason", F.concat_ws("; ", *[F.when(F.col(n), F.lit(n.removeprefix("flag_").upper())) for n in regras]))

# Mesmo id_pedido mais de uma vez no lote: fica a versão mais recente da Bronze.
recente = Window.partitionBy("id_pedido").orderBy(F.col("bronze_ingested_at").desc(), F.col("bronze_source_file").desc())
validos = (validado.filter(F.col("quarantine_reason") == "").withColumn("_rn", F.row_number().over(recente))
                   .filter("_rn = 1").drop("_rn", "quarantine_reason"))
quarentena = validado.filter(F.col("quarantine_reason") != "")

## Gravação
**Silver:** upsert por `id_pedido` (registros existentes que não vieram no lote + lote), particionada por ano/mês do `dt_pedido`. Reexecutar não duplica.
**Quarentena:** append, para manter o histórico de cada reprovação.

In [0]:
def gravar(df, caminho, modo, particao=None):
    w = df.write.format("delta").options(**adls_options).mode(modo).option("overwriteSchema", "true" if modo == "overwrite" else "false")
    (w.partitionBy(*particao) if particao else w).save(caminho)

validos = validos.withColumn("ano", F.year("dt_pedido")).withColumn("mes", F.month("dt_pedido"))
silver_atual = ler_delta(CAMINHO_SILVER)
if silver_atual is not None:
    validos = silver_atual.join(validos.select("id_pedido"), "id_pedido", "left_anti").unionByName(validos)

if total_lote == 0:
    print("Nada novo na Bronze; Silver e quarentena inalteradas.")
else:
    gravar(validos, CAMINHO_SILVER, "overwrite", ["ano", "mes"])
    gravar(quarentena, CAMINHO_QUARENTENA, "append")
    print("Silver e quarentena gravadas.")

## Auditoria

In [0]:
silver, quar = ler_delta(CAMINHO_SILVER), ler_delta(CAMINHO_QUARENTENA)
if silver is None:
    print("Silver ainda não existe.")
else:
    n_quar_lote = quarentena.count() if total_lote else 0
    print(f"Lote: {total_lote} linhas | válidas: {total_lote - n_quar_lote} | quarentena: {n_quar_lote}")
    print(f"Silver: {silver.count()} linhas, {silver.select('id_pedido').distinct().count()} id_pedido distintos")
    if quar is not None:
        print(f"Quarentena acumulada: {quar.count()} linhas")
        display(quar.select([F.sum(F.col(n).cast("int")).alias(n) for n in regras]))
        display(quar.select("id_pedido", "bronze_source_file", "quarantine_reason", "valor_total", "status_pedido", "dt_pedido", "dt_ultima_atualizacao_status"))
    silver.printSchema()

## Resumo
_Preencher após rodar no Databricks: quantas linhas entraram, quantas foram para a quarentena e por qual regra, e o que isso significa para o negócio (ex.: pedidos com valor negativo ou status atualizado antes da criação indicam falha no sistema de origem)._